In [0]:
from pyspark.sql.functions import col, round, abs


catalog_name = "pharma_demo_catalog"
volume_path = f"/Volumes/{catalog_name}/bronze/raw_data"

In [0]:
from pyspark.sql.functions import col, abs, when, lit


df_compliance = spark.table(f"{catalog_name}.silver.erp_inventory_silver") \
    .join(spark.table(f"{catalog_name}.silver.lims_quality_conformed"), "batch_id") \
    .select("batch_id", "product_code", "erp_vial_count", "erp_yield_pct", "performance_status", "row_integrity_hash")

df_compliance.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.gold.manufacturing_batch_compliance")


df_audit = spark.table(f"{catalog_name}.gold.manufacturing_batch_compliance").alias("gold") \
    .join(spark.table(f"{catalog_name}.silver.lims_quality_conformed").alias("lims"), "batch_id") \
    .withColumn("count_variance", col("lims.lims_vial_count") - col("gold.erp_vial_count")) \
    .withColumn("reconciliation_status", when(abs(col("count_variance")) > 0, lit("FLAGGED")).otherwise(lit("RECONCILED"))) \
    .select("batch_id", "lims_vial_count", "erp_vial_count", "count_variance", "reconciliation_status")

df_audit.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.system_reconciliation_audit")


df_compliance.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.manufacturing_batch_compliance")


print("display 1: compliance kpis ")
display(spark.table(f"{catalog_name}.gold.manufacturing_batch_compliance"))

print("\n display 2: reconcilation audit  )")
display(spark.table(f"{catalog_name}.gold.system_reconciliation_audit"))

DISPLAY 1: COMPLIANCE KPIs 


batch_id,product_code,erp_vial_count,erp_yield_pct,performance_status,row_integrity_hash
BATCH_78450,ONCO-INJ-2025-A,9800,98.0,OPTIMAL,7ef0ed848d7cc7123ee7e17e6cc4113e488039bb1f3ec40a613720152e1384f4
BATCH_78451,ONCO-INJ-2025-A,9800,98.0,OPTIMAL,be43774bccd922992964ef8bfa59bbcd1b2c2d211fc90a57068fc69fe9154baf
BATCH_78450,ONCO-INJ-2025-A,9800,98.0,OPTIMAL,7ef0ed848d7cc7123ee7e17e6cc4113e488039bb1f3ec40a613720152e1384f4
BATCH_78451,ONCO-INJ-2025-A,9800,98.0,OPTIMAL,be43774bccd922992964ef8bfa59bbcd1b2c2d211fc90a57068fc69fe9154baf
BATCH_78450,ONCO-INJ-2025-A,9800,98.0,OPTIMAL,7ef0ed848d7cc7123ee7e17e6cc4113e488039bb1f3ec40a613720152e1384f4
BATCH_78451,ONCO-INJ-2025-A,9800,98.0,OPTIMAL,be43774bccd922992964ef8bfa59bbcd1b2c2d211fc90a57068fc69fe9154baf
BATCH_78450,ONCO-INJ-2025-A,9800,98.0,OPTIMAL,7ef0ed848d7cc7123ee7e17e6cc4113e488039bb1f3ec40a613720152e1384f4
BATCH_78451,ONCO-INJ-2025-A,9800,98.0,OPTIMAL,be43774bccd922992964ef8bfa59bbcd1b2c2d211fc90a57068fc69fe9154baf
BATCH_78452,ONCO-INJ-2025-A,10000,95.24,OPTIMAL,8f09137fb17d041a0fd1940e29d6f831a17859bc880d7dd1dc01ac5925280973
BATCH_78452,ONCO-INJ-2025-A,10080,96.0,OPTIMAL,5227689a708519867debd1e8428cc730ede77a7f7247959ad6f640fe2407e38e



 DISPLAY 2: RECONCILIATION AUDIT )


batch_id,lims_vial_count,erp_vial_count,count_variance,reconciliation_status
BATCH_78450,9800,9800,0,RECONCILED
BATCH_78451,9800,9800,0,RECONCILED
BATCH_78450,9800,9800,0,RECONCILED
BATCH_78451,9800,9800,0,RECONCILED
BATCH_78450,9800,9800,0,RECONCILED
BATCH_78451,9800,9800,0,RECONCILED
BATCH_78450,9800,9800,0,RECONCILED
BATCH_78451,9800,9800,0,RECONCILED
BATCH_78452,10000,10000,0,RECONCILED
BATCH_78452,10000,10080,-80,FLAGGED
